# Verbal multiword expressions

Verbal multiword expressions combine a verb with other words, as in
*arveid õiendama*. Their meaning is often not obvious from the individual words.
See the [PARSEME annotation guidelines](https://parsemefr.lis-lab.fr/parseme-st-guidelines/2.0/?page=010_Definitions_and_scope%2F020_Multiword_expressions)
for more background on multiword expressions.

This tagger finds candidates using a lexicon, dependency syntax and clause
boundaries. It does not distinguish idiomatic uses from literal ones, so extra
annotations are expected. Results also depend on the input analyses and lexicon.

The examples use the `verbal_multiword_expr_lexicon` resource from EstNLTK.

In [1]:
from estnltk import Text
from estnltk_neural.taggers import StanzaSyntaxTagger
from estnltk.taggers import VerbalMultiWordExprPreTagger

syntax_tagger = StanzaSyntaxTagger()

## Tag a sentence

In [2]:
text = Text("Ta õiendas arveid.").tag_layer(["morph_analysis", "clauses"])
syntax_tagger.tag(text)

VerbalMultiWordExprPreTagger().tag(text)
text.pre_verbal_multiword_expressions

RelationLayer(name='pre_verbal_multiword_expressions', span_names=('verb', 'comp_1', 'comp_2', 'comp_3', 'comp_4', 'comp_5'), attributes=('expression', 'is_inflected', 'is_central_verb'), relations=[Relation([NamedSpan(verb: 'õiendas'), NamedSpan(comp_1: 'arveid')], [{'expression': 'arveid õiendama', 'is_inflected': True, 'is_central_verb': True}])])

`verb` is the anchor; `comp_1` ... `comp_5` are the other words.
`expression` shows the lexicon entry. `is_inflected` indicates whether the other
words may inflect (`True`) or must keep their fixed forms (`False`). The verb
may change form in both cases. `is_central_verb` indicates that no verb occurs
above the anchor in its clause.

## Infinitives and participles

Use `include_nonfinite=True` to include non-finite anchors.

In [3]:
text = Text("Ta tahtis arveid õiendada.").tag_layer(["morph_analysis", "clauses"])
syntax_tagger.tag(text)

VerbalMultiWordExprPreTagger(include_nonfinite=True).tag(text)
text.pre_verbal_multiword_expressions

RelationLayer(name='pre_verbal_multiword_expressions', span_names=('verb', 'comp_1', 'comp_2', 'comp_3', 'comp_4', 'comp_5'), attributes=('expression', 'is_inflected', 'is_central_verb'), relations=[Relation([NamedSpan(verb: 'õiendada'), NamedSpan(comp_1: 'arveid')], [{'expression': 'arveid õiendama', 'is_inflected': True, 'is_central_verb': False}])])

## Particle verbs

`include_compound_verbs=True` adds syntax-derived particle verbs and the
`is_compound_verb` attribute.

In [4]:
text = Text("Poiss annab palli tagasi.").tag_layer(["morph_analysis", "clauses"])
syntax_tagger.tag(text)

VerbalMultiWordExprPreTagger(include_compound_verbs=True).tag(text)
text.pre_verbal_multiword_expressions

RelationLayer(name='pre_verbal_multiword_expressions', span_names=('verb', 'comp_1', 'comp_2', 'comp_3', 'comp_4', 'comp_5'), attributes=('expression', 'is_inflected', 'is_central_verb', 'is_compound_verb'), relations=[Relation([NamedSpan(verb: 'annab'), NamedSpan(comp_1: 'tagasi')], [{'expression': 'tagasi andma', 'is_inflected': None, 'is_central_verb': True, 'is_compound_verb': True}])])

## Copular expressions

In "on abiks", syntax attaches "on" below "abiks".
By default, `include_copula_heads=True` also searches the predicate and its
subtree within the same clause. Set it to `False` to disable this expansion.

In [5]:
text = Text("Ta on abiks.").tag_layer(["morph_analysis", "clauses"])
syntax_tagger.tag(text)

VerbalMultiWordExprPreTagger().tag(text)
text.pre_verbal_multiword_expressions

RelationLayer(name='pre_verbal_multiword_expressions', span_names=('verb', 'comp_1', 'comp_2', 'comp_3', 'comp_4', 'comp_5'), attributes=('expression', 'is_inflected', 'is_central_verb'), relations=[Relation([NamedSpan(verb: 'on'), NamedSpan(comp_1: 'abiks')], [{'expression': 'abiks olema', 'is_inflected': True, 'is_central_verb': True}])])

## Use another lexicon

Save your entries as a JSON list and pass the file with
`VerbalMultiWordExprPreTagger(lexicon_path="my_lexicon.json")`. For example:

```json
[
  {
    "expression": "aru saama",
    "is_inflected": false,
    "components": [
      {"form": "aru", "lemmas": ["aru"], "match_by_lemma": false, "is_anchor": false},
      {"form": "saama", "lemmas": ["saama"], "match_by_lemma": true, "is_anchor": true}
    ]
  }
]
```

Mark one component as the anchor verb. `match_by_lemma` selects lemma matching;
otherwise the tagger matches the listed `form`.

To generate the JSON from CSV lists, use the
[lexicon preparation notebook](https://github.com/kaarelkytt/estnltk-model-training/blob/1b65a52df1c4bf8b2f91ed42cfc0ee72685afdd3/multiword_expr_tools/create_verbal_mwe_json.ipynb).